# WP9: fine-tune the RouteChatbot classifier on a Kaggle T4

Owner: Person C. Runs `finetune/prepare.py`, `finetune/train.py` and `finetune/select_checkpoint.py` from the repository with the settings in `finetune/config.yaml` (handbook WP9).

**Before you run**
1. Settings, Accelerator: GPU T4 x1. Internet: On.
2. D1 is in the repository at `data/d1/` (merged 23 Nov), so no extra input is needed. Only `train.jsonl` and `val.jsonl` are read; `test.jsonl` is never used here.
3. No secrets needed: the repository is public. (If it is ever made private, add a `GITHUB_TOKEN` Kaggle secret with read access and the clone cell will use it.)
4. Run all. One full run is about 3 epochs over 1,600 records, roughly 20 to 30 minutes on a T4. The sweep (4 runs) and the learning curve (4 runs) are separate cells you turn on with the flags below.

The adapter it produces is shared as a private Kaggle dataset; it is never committed (`finetune/artifacts/.gitignore`).

In [ ]:
RUN_SWEEP = False          # step 5: r in {8, 16} x lr in {1e-4, 2e-4}
RUN_LEARNING_CURVE = False # step 6: 200, 400, 800, 1600 records with the chosen setting
RUN_REPRO_CHECK = False    # T-9-03: train the chosen setting twice with seed 42 and compare
CHOSEN_R, CHOSEN_LR = 16, 2e-4
D1_DIR = "data/d1"   # inside the cloned repository: train.jsonl and val.jsonl
REPO = "pranaycv/RouteChatbot"
BRANCH = "main"

In [ ]:
%pip install -q -U "transformers>=4.56" "peft>=0.13" "trl>=0.20" "datasets>=3.0" accelerate pydantic pyyaml python-dotenv

In [ ]:
import os, subprocess, json, pathlib
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # the repository is public, so no token is needed
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not pathlib.Path("RouteChatbot").exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url], check=True)
os.chdir("RouteChatbot")
os.environ["PYTHONPATH"] = os.getcwd()
print(subprocess.run(["git", "log", "--oneline", "-n", "1"], capture_output=True, text=True).stdout)

In [ ]:
# Step 2: prepare. Same system prompt and serialisation as runtime (T-9-01).
!python finetune/prepare.py --in {D1_DIR}/train.jsonl --out finetune/data/train.jsonl
!python finetune/prepare.py --in {D1_DIR}/val.jsonl   --out finetune/data/val.jsonl
!python finetune/train.py --train finetune/data/train.jsonl --val finetune/data/val.jsonl --out finetune/artifacts/dry --dry-run | head -30

In [ ]:
# Step 3: one training run with the handbook settings.
!python finetune/train.py --train finetune/data/train.jsonl --val finetune/data/val.jsonl --out finetune/artifacts/run_r{CHOSEN_R}_lr{CHOSEN_LR}

In [ ]:
# Step 4: pick the checkpoint by joint exact match on validation, and copy it to the adapter folder WP2 loads.
!python finetune/select_checkpoint.py --run finetune/artifacts/run_r{CHOSEN_R}_lr{CHOSEN_LR} --val finetune/data/val.jsonl --copy-to finetune/artifacts/adapter

In [ ]:
# Step 5: the small sweep. Four runs; compare select_results.json across them.
if RUN_SWEEP:
    import yaml
    grid = yaml.safe_load(open("finetune/config.yaml"))["sweep"]
    for r in grid["r"]:
        for lr in grid["learning_rate"]:
            out = f"finetune/artifacts/sweep_r{r}_lr{lr}"
            !python finetune/train.py --train finetune/data/train.jsonl --val finetune/data/val.jsonl --out {out} --r {r} --lr {lr}
            !python finetune/select_checkpoint.py --run {out} --val finetune/data/val.jsonl
    for p in sorted(pathlib.Path("finetune/artifacts").glob("sweep_*/select_results.json")):
        res = json.load(open(p)); print(p.parent.name, res["best"], round(res["results"][res["best"]]["joint_exact_match"], 3))

In [ ]:
# Step 6: the learning curve. Validation joint exact match against training-set size.
if RUN_LEARNING_CURVE:
    import yaml
    sizes = yaml.safe_load(open("finetune/config.yaml"))["learning_curve"]["sizes"]
    curve = {}
    for n in sizes:
        out = f"finetune/artifacts/curve_{n}"
        !python finetune/train.py --train finetune/data/train.jsonl --val finetune/data/val.jsonl --out {out} --r {CHOSEN_R} --lr {CHOSEN_LR} --subset {n}
        !python finetune/select_checkpoint.py --run {out} --val finetune/data/val.jsonl
        res = json.load(open(f"{out}/select_results.json")); curve[n] = res["results"][res["best"]]["joint_exact_match"]
    json.dump(curve, open("finetune/artifacts/learning_curve.json", "w"), indent=2)
    import matplotlib.pyplot as plt
    plt.plot(list(curve), list(curve.values()), marker="o"); plt.xlabel("training records"); plt.ylabel("validation joint exact match"); plt.title("WP9 learning curve"); plt.grid(True)
    plt.savefig("finetune/artifacts/learning_curve.png", dpi=150, bbox_inches="tight"); print(curve)

In [ ]:
# T-9-03: reproducibility. Same seed twice; validation joint exact match must agree within 1 point.
if RUN_REPRO_CHECK:
    scores = []
    for i in (1, 2):
        out = f"finetune/artifacts/repro_{i}"
        !python finetune/train.py --train finetune/data/train.jsonl --val finetune/data/val.jsonl --out {out} --r {CHOSEN_R} --lr {CHOSEN_LR}
        !python finetune/select_checkpoint.py --run {out} --val finetune/data/val.jsonl
        res = json.load(open(f"{out}/select_results.json")); scores.append(res["results"][res["best"]]["joint_exact_match"])
    print("T-9-03", scores, "PASS" if abs(scores[0] - scores[1]) <= 0.01 else "FAIL")

In [ ]:
# Step 7 and 8: package the adapter and record the GPU cost for WP11.
import shutil
shutil.make_archive("/kaggle/working/routechatbot_adapter", "zip", "finetune/artifacts/adapter")
runs = sorted(pathlib.Path("finetune/artifacts").glob("*/run.json"))
total_hours = sum(json.load(open(p))["gpu_hours"] for p in runs)
rate = __import__("yaml").safe_load(open("finetune/config.yaml"))["cost"]["t4_usd_per_hour"]
print(f"runs: {len(runs)}  GPU hours: {total_hours:.2f}  cost at ${rate}/h: ${total_hours * rate:.2f}")
print("Adapter zip: /kaggle/working/routechatbot_adapter.zip  (upload as a private Kaggle dataset, then set CLASSIFIER_ADAPTER_DIR)")